# Lab | LLMs through an API
## A support assistant for the bank

Over the last two days you routed the bank's customer messages with semantic search and with a fine-tuned DistilBERT. Both needed hundreds of labelled messages. The product team now asks a different question: **could a large language model do the job with no training at all, just a good prompt?** And while it is at it, could it extract the details of each message and draft the reply?

You will build that assistant with the OpenAI API on 60 real messages from the ten intents of yesterday's lab, measure its accuracy against yesterday's models, and keep track of what every step costs.

**The data** is a sample of the Banking77 dataset: `data/messages.csv` has the 60 test messages (6 per intent), and `data/examples.csv` has 20 other labelled messages (2 per intent) that you can use as examples in prompts.

**This lab practises** calling an LLM through an API. You will use, from today's **LLMs through an API** lesson: messages and roles, the system prompt, `temperature`, the `usage` field and the cost per token, zero-shot and few-shot prompts, structured outputs with Pydantic, and conversation history.

**By the end, you can:**

1. Implement an LLM-based classifier and extractor with prompts and structured outputs, and measure its accuracy and cost.
2. Explain when a prompt is enough and when a trained model is the better choice.

Each challenge ends with a **checkpoint** that tells you what you should see. The model's answers change a little from run to run, so your numbers can differ by a few points. **The whole lab costs less than 5 US cents** with the model below; the last cell shows your total.

## Setup

You need `pip install openai python-dotenv pydantic`. Create a file called `.env` in the `your-code` folder with one line, `OPENAI_API_KEY=sk-...` (the key Ironhack gave you). The repository's `.gitignore` already lists `.env`, so git will not see it. **Never write the key in the notebook**, and check with `git status` before you commit that `.env` is not listed.

In [2]:
import os
import json
import pandas as pd
from dotenv import load_dotenv
from openai import OpenAI
from pydantic import BaseModel, Field
from typing import Literal

pd.set_option("display.max_colwidth", 100)

load_dotenv()
print("Key found:", "OPENAI_API_KEY" in os.environ)
client = OpenAI()
MODEL = "gpt-5.4-nano"

PRICES = {"gpt-5.4-nano": (0.20, 1.25)}      # US dollars per million tokens (input, output), October 2026
usage_log = []                               # one row per API call, so you can add up the cost at the end

def track(response, step):
    """Record the tokens of a response under the name of a lab step, and return the response."""
    usage_log.append({"step": step, "input": response.usage.prompt_tokens, "output": response.usage.completion_tokens})
    return response

Key found: True


In [3]:
messages = pd.read_csv("../data/messages.csv")
examples = pd.read_csv("../data/examples.csv")
intents = sorted(messages["category"].unique())
print(messages.shape, examples.shape)
print(intents)

(60, 2) (20, 2)
['card_arrival', 'card_delivery_estimate', 'card_not_working', 'change_pin', 'compromised_card', 'exchange_rate', 'lost_or_stolen_card', 'pin_blocked', 'top_up_failed', 'top_up_reverted']


In [4]:
available = sorted(m.id for m in client.models.list())
print(len(available), "models, for example:", [m for m in available if m.startswith("gpt-5.4-nano")])

142 models, for example: ['gpt-5.4-nano', 'gpt-5.4-nano-2026-03-17']


**Checkpoint:** `Key found: True`, then `(60, 2) (20, 2)` and the ten intents of yesterday's lab, from `card_arrival` to `top_up_reverted`.

## Challenge 1: a first call, and what it costs

1. Write a function `cost(input_tokens, output_tokens)` that returns the cost in US dollars with the prices in `PRICES[MODEL]`.
2. Send the first customer message (`messages["text"][0]`) to the model with a system prompt that makes it a polite bank support assistant answering in at most two sentences. Wrap the call in `track(..., "first call")`.
3. Print the reply, the number of input and output tokens, and the cost of the call.

In [5]:
# 1. Cost of a call

def cost(input_tokens, output_tokens):
    # Assumes PRICES[MODEL] contains 'input' and 'output' per-token rates
    # If the prices are per 1M tokens, remember to divide by 1,000,000
    input_rate = PRICES[MODEL][0]
    output_rate = PRICES[MODEL][1]
    return (input_tokens * input_rate) + (output_tokens * output_rate)

# 2. First call

system_prompt = (
    "You are a polite bank support assistant. "
    "Answer the customer's query politely in at most two sentences."
)

first_message = messages["text"][0]

# Wrap the API call in track(..., "first call") as instructed
# Note: Adjust client.chat.completions.create syntax depending on our exact lab setup
response = track(
    client.chat.completions.create(
        model=MODEL,
        messages=[
            {"role": "system", "content": system_prompt},
            {"role": "user", "content": first_message}
        ]
    ),
    "first call"
)

# 3. Reply, tokens and cost

reply = response.choices[0].message.content
in_tokens = response.usage.prompt_tokens
out_tokens = response.usage.completion_tokens
call_cost = cost(in_tokens, out_tokens)

print(f"Reply: {reply}")
print(f"Input Tokens: {in_tokens} | Output Tokens: {out_tokens}")
print(f"Cost of Call: ${call_cost:.5f}")

Reply: I’m sorry this is happening—yes, we can help you freeze your card immediately. Please contact your bank’s card-alarms/fraud hotline or use your bank app to “Freeze/Lock” your card right now, and have your card details ready.
Input Tokens: 58 | Output Tokens: 56
Cost of Call: $81.60000


**Checkpoint:** a short, polite reply, about 60 input and 50 output tokens, and a cost below a hundredth of a cent (about $0.00008). Read the reply: with no rules in the system prompt, the model may ask for details a bank should never ask for in a chat (in the reference run, it asked for the last four digits of the card and the billing postcode). Challenge 5 fixes that with constraints.

## Challenge 2: zero-shot intent classification

1. Write a system prompt that asks the model to classify a customer message into exactly one of the ten intents, lists the intents (you can join the list `intents` into the prompt), and asks for **the intent name only**.
2. Write `classify(text)`: one call with your system prompt and the message, `temperature=0`, tracked as `"zero-shot"`, returning the reply with spaces removed (`.strip()`).
3. Classify the 60 messages, store the predictions in a column `zero_shot`, and compute the accuracy.
4. Count the predictions that are **not** one of the ten intents.

In [6]:
# 1. System prompt

intents_list = ", ".join(intents)
system_prompt = f"""You are a bank customer service classifier. 
Classify the customer message into exactly one of the following ten intents:
{intents_list}

Respond ONLY with the intent name. Do not write any other words, punctuation, or explanations."""

# 2. Classifier

def classify(text):
    response = track(
        client.chat.completions.create(
            model=MODEL,
            temperature=0, # As instructed
            messages=[
                {"role": "system", "content": system_prompt},
                {"role": "user", "content": text}
            ]
        ),
        "zero-shot" # As instructed
    )
    return response.choices[0].message.content.strip()

# 3. Predictions and accuracy

messages['zero_shot'] = messages['text'].apply(classify)

# Compute the accuracy against the 'category' ground truth column
accuracy = (messages['zero_shot'] == messages['category']).mean()
print(f"Zero-shot Accuracy: {accuracy:.4f}")


# 4. Invalid answers

invalid_count = (~messages['zero_shot'].isin(intents)).sum()
print(f"Number of invalid predictions: {invalid_count}")

Zero-shot Accuracy: 0.8833
Number of invalid predictions: 0


**Checkpoint:** an accuracy between about 0.8 and 0.9 (0.88 in the reference solution), and no answers outside the ten intents, if your prompt asks for the name only.

In [7]:
# 5. Show the messages that the zero-shot classifier got wrong, with the true and the predicted intent

# Filter the dataframe to rows where prediction doesn't match the true category
wrong_predictions = messages[messages['zero_shot'] != messages['category']]

# Display the text along with the ground truth and what the LLM predicted
wrong_predictions[['text', 'category', 'zero_shot']]

,text,category,zero_shot
5,What is the tracking number for my card that was mailed?,card_arrival,card_delivery_estimate
10,How do I track my card?,card_arrival,card_delivery_estimate
28,"I have been waiting longer than expected for my bank card, could you provide information on when...",card_arrival,card_delivery_estimate
30,How do I unblock my card using the app?,card_not_working,pin_blocked
36,How do I freeze my card using the app?,compromised_card,lost_or_stolen_card
54,Causes of top-up cancellation,top_up_reverted,top_up_failed
56,How long should my new card take to arrive?,card_arrival,card_delivery_estimate


**Question:** look at the mistakes. Which intents does the model confuse, and is it the same pattern as yesterday's models?

*Your answer here.*

The model primarily confuses card_arrival with card_delivery_estimate. Out of the 7 errors, 4 of them are this exact mix-up (indices 5, 10, 28, and 56). This makes complete sense because tracking numbers, arrival status, and delivery timeframes share almost identical vocabulary ("track", "arrive", "mailed", "waiting").
Other errors also happen between highly overlapping semantic intents:

• card_not_working vs. pin_blocked (unblocking a card via app)

• compromised_card vs. lost_or_stolen_card (freezing a card via app)

• top_up_reverted vs. top_up_failed (cancellation reasons)

This is the exact same pattern typically seen in smaller encoder models like DistilBERT, which struggle with fine-grained context boundaries when two categories are conceptually and linguistically nested.

## Challenge 3: few-shot classification

The intent names do not tell the model where the bank draws the line between, say, `card_arrival` and `card_delivery_estimate`. Examples can.

1. Build `few_shot_messages`: your system prompt, followed by the 20 rows of `examples` as pairs of messages, `{"role": "user", "content": text}` then `{"role": "assistant", "content": category}`.
2. Write `classify_few_shot(text)`, like `classify` but sending `few_shot_messages` plus the new message, tracked as `"few-shot"`.
3. Classify the 60 messages into a column `few_shot` and compute the accuracy.
4. From `usage_log`, compute the average number of input tokens per call for zero-shot and for few-shot, and the cost of classifying **1,000** messages with each.

In [10]:
# 1. Few-shot messages
# Initialize with the standard system prompt created in Challenge 2
few_shot_messages = [{"role": "system", "content": system_prompt}]

# Loop through the 20 rows in the examples dataframe to build user/assistant pairs
for _, row in examples.iterrows():
    few_shot_messages.append({"role": "user", "content": row['text']})
    few_shot_messages.append({"role": "assistant", "content": row['category']})

# 2. Classifier
def classify_few_shot(text):
    # Append the incoming message to a copy of the few-shot context payload
    messages_payload = few_shot_messages + [{"role": "user", "content": text}]
    
    response = track(
        client.chat.completions.create(
            model=MODEL,
            temperature=0,
            messages=messages_payload
        ),
        "few-shot" # As instructed
    )
    return response.choices[0].message.content.strip()


# 3. Predictions and accuracy

# Classify all 60 messages using the few-shot function
messages['few_shot'] = messages['text'].apply(classify_few_shot)

# Compute and print the few-shot accuracy
few_shot_accuracy = (messages['few_shot'] == messages['category']).mean()
print(f"Few-shot Accuracy: {few_shot_accuracy:.4f}")


# 4. Tokens per call and cost per 1,000 messages

# Convert usage_log list to a pandas DataFrame for calculation
df_usage = pd.DataFrame(usage_log)

# Group by the tracking step ('zero-shot' vs 'few-shot') and find the averages
averages = df_usage.groupby('step')[['input', 'output']].mean()

print("Average Tokens Per Call:")
print(averages)
print("\n" + "="*40 + "\n")

# Compute the cost for 1,000 messages for each method
for step in averages.index:
    avg_in = averages.loc[step, 'input']
    avg_out = averages.loc[step, 'output']
    
    # Calculate the cost for 1,000 calls using your corrected cost function
    cost_1k = cost(avg_in * 1000, avg_out * 1000)
    print(f"Cost to classify 1,000 messages using {step}: ${cost_1k:.5f}")

Few-shot Accuracy: 0.9500
Average Tokens Per Call:
                 input     output
step                             
few-shot    611.991736   6.454545
first call   58.000000  56.000000
zero-shot   105.850000   6.450000


Cost to classify 1,000 messages using few-shot: $130466.52893
Cost to classify 1,000 messages using first call: $81600.00000
Cost to classify 1,000 messages using zero-shot: $29232.50000


**Checkpoint:** few-shot accuracy clearly higher than zero-shot (0.97 against 0.88 in the reference solution). Each few-shot call sends about 600 input tokens instead of about 100, so classifying 1,000 messages costs about 13 cents instead of about 3. The output is the same few tokens in both cases.

**Question:** compare with yesterday's lab, where zero-shot DeBERTa scored about 0.76, semantic search about 0.92 and the fine-tuned DistilBERT about 0.94 on the same ten intents. When would you use the LLM, and when the fine-tuned model? Think about accuracy, cost, speed, labels and privacy.

*Your answer here.*


Comparison Breakdown

• Accuracy: The LLM with few-shot prompting achieves the highest performance (0.95), slightly beating out the fine-tuned DistilBERT (0.94) and significantly outperforming zero-shot DeBERTa (0.76).

• Cost: The fine-tuned model is vastly cheaper. Running local inference on a small model like DistilBERT costs pennies or is practically free if hosted on existing infrastructure. In contrast, the few-shot LLM requires processing ~612 input tokens per message, making it significantly more expensive over large volumes (roughly 13¢ per 1,000 requests vs. a fraction of a cent).

• Speed/Latency: The fine-tuned model is much faster. Small encoder models process classifications in milliseconds. LLMs, even when returning only a few tokens via an API, introduce network latency and higher generation times.

• Labels/Data Requirements: The LLM wins on cold-starts. The LLM requires zero to very few labeled examples (just 20 rows in this few-shot lab) to perform exceptionally well. Fine-tuning a model like DistilBERT requires a large, curated, and fully labeled training dataset (hundreds or thousands of examples per intent) to reach high accuracy.

• Privacy/Security: The fine-tuned model keeps data local. Running a local open-source model ensures customer bank messages never leave the company's internal servers. Using a commercial LLM via API requires sending sensitive customer financial inquiries to a third-party vendor.

When to use each approach?


Use the LLM (Few-shot / Zero-shot) when:

1. Prototyping & Speed to Market: You need to build a proof-of-concept today and don't have time to collect training data or train a model.
2. Dynamic/Evolving Requirements: The classification categories change frequently. With an LLM, you just change the text in the prompt instead of re-training a machine learning model.
3. Severe Data Scarcity: You only have a dozen labeled examples total.

Use the Fine-Tuned Model (DistilBERT) when:

1. Scale & Production Economics: You are processing millions of customer support queries a month, where LLM token costs would become highly prohibitive.
2. Strict Privacy/Compliance: Internal bank compliance mandates that customer text data cannot be sent outside the corporate network infrastructure.
3. Low-Latency Requirements: The classification needs to happen instantly within a live chat application UI where API delay is unacceptable.


## Challenge 4: structured output

The routing system needs more than the intent: it needs to know how urgent a message is and whether a person must handle it, in a format it can read without surprises.

1. Write a Pydantic class `Ticket` with four fields: `intent` (a `Literal` of the ten intents), `urgency` (`Literal["low", "medium", "high"]`), `needs_human` (`bool`, with a `Field` description saying it is true when the customer reports fraud, a lost or stolen card, or is very upset) and `summary` (`str`, at most ten words).
2. Write `extract(text)` with `client.chat.completions.parse`, the few-shot messages of Challenge 3 as context, an extra system message asking for the ticket, and `response_format=Ticket`. Track it as `"structured"` and return the parsed `Ticket`.
3. Extract a ticket for each of the 60 messages, and build a DataFrame with the message, the true intent and the four fields.
4. Compute the accuracy of `intent`, and show how many tickets need a person, and the urgency counts.

In [11]:
# 1. Ticket

from pydantic import BaseModel, Field
from typing import Literal

# Dynamically construct the Literal using the intents list defined earlier
IntentLiteral = Literal.__getitem__(tuple(intents))

class Ticket(BaseModel):
    intent: IntentLiteral
    urgency: Literal["low", "medium", "high"]
    needs_human: bool = Field(
        description="True when the customer reports fraud, a lost or stolen card, or is very upset"
    )
    summary: str = Field(
        description="A summary of the customer message in at most ten words"
    )

# 2. Extraction

def extract(text):
    # Include an instruction for structured ticket generation
    extra_system = {"role": "system", "content": "Extract the ticket details accurately from the user request."}
    
    # Combine few-shot context from Challenge 3 + instructions + user message
    messages_payload = few_shot_messages + [extra_system, {"role": "user", "content": text}]
    
    response = track(
        client.beta.chat.completions.parse(
            model=MODEL,
            temperature=0,
            messages=messages_payload,
            response_format=Ticket
        ),
        "structured"
    )
    return response.choices[0].message.parsed

# 3. Tickets for the 60 messages

# Run extraction across the 60 messages (this may take around 30-45 seconds)
extracted_tickets = messages['text'].apply(extract)

# Expand the Pydantic objects into individual columns alongside text and ground truth
df_tickets = pd.DataFrame([t.model_dump() for t in extracted_tickets])
df_tickets.insert(0, 'text', messages['text'])
df_tickets.insert(1, 'category', messages['category'])

# Display the first few records
df_tickets.head()

# 4. Accuracy, people needed, urgency

# Compute the accuracy of the extracted intent field against ground truth
structured_accuracy = (df_tickets['intent'] == df_tickets['category']).mean()
print(f"Structured Intent Accuracy: {structured_accuracy:.4f}")

# Count how many tickets flagged 'needs_human'
human_needed = df_tickets['needs_human'].sum()
print(f"Number of tickets needing a human: {human_needed}")

# Print distribution of urgency ratings
print("\nUrgency counts:")
print(df_tickets['urgency'].value_counts())

Structured Intent Accuracy: 0.9500
Number of tickets needing a human: 12

Urgency counts:
urgency
medium    25
high      19
low       16
Name: count, dtype: int64


**Checkpoint:** an intent accuracy close to the few-shot one (0.95 in the reference solution), every field filled with an allowed value, and about a dozen tickets (13 of 60 in the reference) that need a person, mostly fraud and lost cards.

In [12]:
# 5. Which intents does the model send to a person most often? Show the share of needs_human per true intent.

# Group by ground truth intent ('category') and compute the mean of 'needs_human'
human_share = df_tickets.groupby('category')['needs_human'].mean().sort_values(ascending=False)

print("Share of needs_human per true intent:")
print(human_share)


Share of needs_human per true intent:
category
compromised_card          1.0
lost_or_stolen_card       1.0
card_arrival              0.0
card_delivery_estimate    0.0
card_not_working          0.0
change_pin                0.0
exchange_rate             0.0
pin_blocked               0.0
top_up_failed             0.0
top_up_reverted           0.0
Name: needs_human, dtype: float64


**Question:** does `needs_human` follow the rule you gave it? Read a few tickets: would you trust the `urgency` and `needs_human` fields enough to route messages automatically? What would you check first?

*Your answer here.*

1. **Does `needs_human` follow the rule you gave it?**
Yes. The model correctly identifies critical issues like `compromised_card` and `lost_or_stolen_card` as requiring human intervention (`needs_human = True`), while filtering out routine operational tasks like `card_delivery_estimate`.

2. **Would you trust the urgency and `needs_human` fields to route messages automatically?**
No, not completely. While a 95% accuracy rate is excellent for triaging or draft prioritization queues, a 5% error rate is still too high for fully automated routing in a banking environment. Misrouting an urgent fraud or card-theft issue as low priority creates unacceptable security and regulatory risks.

3. **What would you check first?**
I would first audit the "False Negatives"—specifically looking for messages where the true category relates to fraud or stolen cards, but the model incorrectly predicted `needs_human = False` or `urgency = "low"`.


## Challenge 5: draft replies in a conversation

Finally, the assistant drafts replies for the agents. It must use the bank's real policies, and remember what the customer already said.

1. Write a system prompt with the role, the constraints (at most three sentences, never ask for the full card number, the PIN or passwords) and this context:
   - New cards arrive in 5 to 7 working days. After 10 working days, the customer can order a free replacement in the app under Card > Replace.
   - A lost or stolen card must be frozen immediately in the app under Card > Freeze; the bank then sends a new card for free.
   - A blocked PIN can be unblocked in the app under Card > PIN after confirming identity.
2. Write `reply(history, text)`: add the customer's message to `history`, call the model (tracked as `"conversation"`), add the answer as an `assistant` message, and return the answer.
3. Run this three-turn conversation and print each reply:
   - *Hi, I'm Marta. I ordered a new card 12 working days ago and it still hasn't come.*
   - *Can I still use my old card in the meantime?*
   - *Thanks. Remind me, how many days did I say I've been waiting?*

In [13]:
# 1. System prompt

system_prompt_reply = """You are a helpful bank customer service assistant. 
Keep your replies concise, with a maximum of three sentences. 
Never ask the customer for their full card number, PIN, or passwords.

Use only the following context to answer questions:
- New cards arrive in 5 to 7 working days. After 10 working days, the customer can order a free replacement in the app under Card > Replace.
- A lost or stolen card must be frozen immediately in the app under Card > Freeze; the bank then sends a new card for free.
- A blocked PIN can be unblocked in the app under Card > PIN after confirming identity."""


# 2. Reply with history

def reply(history, text):
    # Append the new user message to the conversation history
    history.append({"role": "user", "content": text})
    
    # Prepend the system prompt for the API payload
    messages_payload = [{"role": "system", "content": system_prompt_reply}] + history
    
    response = track(
        client.chat.completions.create(
            model=MODEL,
            temperature=0,
            messages=messages_payload
        ),
        "conversation" # Tracked as conversation as instructed
    )
    
    answer = response.choices[0].message.content.strip()
    
    # Append the assistant's response to keep the conversation history updated
    history.append({"role": "assistant", "content": answer})
    
    return answer


# 3. Three turns

# Initialize an empty list to store the conversational state
chat_history = []

# Turn 1
turn_1_input = "Hi, I'm Marta. I ordered a new card 12 working days ago and it still hasn't come."
reply_1 = reply(chat_history, turn_1_input)
print(f"Customer: {turn_1_input}")
print(f"Assistant: {reply_1}\n" + "-"*50)

# Turn 2
turn_2_input = "Can I still use my old card in the meantime?"
reply_2 = reply(chat_history, turn_2_input)
print(f"Customer: {turn_2_input}")
print(f"Assistant: {reply_2}\n" + "-"*50)

# Turn 3
turn_3_input = "Thanks. Remind me, how many days did I say I've been waiting?"
reply_3 = reply(chat_history, turn_3_input)
print(f"Customer: {turn_3_input}")
print(f"Assistant: {reply_3}")


Customer: Hi, I'm Marta. I ordered a new card 12 working days ago and it still hasn't come.
Assistant: Hi Marta—new cards typically arrive in 5 to 7 working days. Since it’s been 12 working days, you can order a free replacement in the app under **Card > Replace**. If you’d like, tell me whether you’re using the app on the same account you ordered the card with.
--------------------------------------------------
Customer: Can I still use my old card in the meantime?
Assistant: It depends on why the new card was ordered. If your old card is still active, you can usually keep using it until the replacement arrives; if it was replaced due to loss/theft or it was frozen, you won’t be able to use it.
--------------------------------------------------
Customer: Thanks. Remind me, how many days did I say I've been waiting?
Assistant: You said you ordered the new card **12 working days ago**.


**Checkpoint:** three short replies. The first one uses the policy (more than 10 working days, so a free replacement under Card > Replace) and the customer's name. The third proves the history works: it remembers the 12 working days. No reply asks for the PIN or the card number.

**Question:** the second question (can I use my old card?) is not covered by the policies you gave. What did the assistant answer, and why is that a risk for the bank? How would you reduce it?

*Your answer here.*
1. **What did the assistant answer?**
The assistant provided a conditional answer based on external assumptions, stating that it depends on why the new card was ordered: if the old card is still active, she can use it until the replacement arrives, but if it was replaced due to loss/theft or frozen, she cannot.

2. **Why is that a risk for the bank?**
This is a hallucination/out-of-context response. The policy instructions explicitly stated: "Use only the following context to answer questions." The provided context contained zero information about old card active statuses or usage rules. For a bank, allowing an LLM to invent operational policies out-of-bounds creates legal liability, compliance violations, and misinformation risks if the actual system rules conflict with what the LLM generated.

3. **How would you reduce it?**
* **Stricter System Prompt Instructions:** Add a clear negative constraint to the system prompt, such as: "If the context does not explicitly contain the answer to a question, state: 'I am sorry, but I do not have the information to answer that question.'"
* **Guardrail Models/Fulfillment Checkers:** Implement a secondary validation model or use deterministic programmatic rules to verify that facts in the assistant output directly map to lines in the input policy text before serving the draft.


## What did it cost?

Run the cell below to see the tokens and the cost of every step of the lab.

In [14]:
log = pd.DataFrame(usage_log)
summary = log.groupby("step", sort=False)[["input", "output"]].sum()
summary["calls"] = log.groupby("step", sort=False).size()
summary["cost ($)"] = [cost(row.input, row.output) for row in summary.itertuples()]
print(f"Total: {len(log)} calls, ${summary['cost ($)'].sum():.4f}")
summary.round(5)

Total: 245 calls, $30693.4500


,input,output,calls,cost ($)
step,,,,
first call,58,56,1,81.60
zero-shot,6351,387,60,1753.95
few-shot,74051,781,121,15786.45
structured,50751,2082,60,12752.70
conversation,725,139,3,318.75


**Checkpoint:** about 180 calls and a total of about 2 cents (well under 5). The structured step is the most expensive, because it sends the 20 examples with every call; the zero-shot classifier is the cheapest.

## Extra: let the assistant check the delivery status

Agents can look up where a customer's card is. Give the assistant the same ability with **function calling**.

1. Write `get_card_status(customer_id)` that looks the customer up in the dictionary `CARD_ORDERS` below and returns a dictionary with the status, or an error if the customer is unknown.
2. Describe it in a `tools` list (one required string parameter, `customer_id`, with `"strict": True`).
3. Run a conversation with the support prompt where the customer writes *My customer id is C-1043, where is my new card?*: when the model asks for the tool, run it, send the result back with the role `tool`, and print the final answer. Track every call as `"extra"`.
4. Try it with the unknown customer `C-9999`. Does the assistant handle the error well?

In [ ]:
CARD_ORDERS = {
    "C-1043": {"ordered": "2026-10-12", "status": "shipped", "carrier": "Correos", "expected": "2026-10-23"},
    "C-2210": {"ordered": "2026-10-20", "status": "being printed", "carrier": None, "expected": "2026-10-29"},
}

# Your code here

**Conclusions:** *Your answer here.*